In [1]:
# =============
# 环境、路径与参数
# =============

import json
from pathlib import Path
from PIL import Image

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torchvision import transforms

from fire_models import load_fire_model_from_checkpoint


CURRENT_DIR = Path(".").resolve()
PROJECT_ROOT = Path("..").resolve()

IMAGE_DIR = PROJECT_ROOT / "images"

SOURCE_SPLIT_DIR = PROJECT_ROOT / "fire_splits"
VAL_CSV = SOURCE_SPLIT_DIR / "val_fixed_real_only.csv"
TEST_CSV = SOURCE_SPLIT_DIR / "test_fixed_real_only.csv"

MODEL_SAVE_DIR = CURRENT_DIR / "models"
ATTENTION_RESULT_DIR = CURRENT_DIR / "attention_results"
ATTENTION_RESULT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLD_JSON = CURRENT_DIR / "test_results" / "threshold_search" / "best_threshold_auto.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMG_SIZE = 384
FINETUNE_MODE = "layer4_last"

DROPOUT = 0.35
MAX_POOL_SCALE = 0.5

MODEL_PREFIX = "fire_convnext_tiny_attention"

DEFAULT_THRESHOLD = 0.6

print("DEVICE:", DEVICE)
print("CURRENT_DIR:", CURRENT_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGE_DIR:", IMAGE_DIR)
print("SOURCE_SPLIT_DIR:", SOURCE_SPLIT_DIR)
print("MODEL_SAVE_DIR:", MODEL_SAVE_DIR)
print("ATTENTION_RESULT_DIR:", ATTENTION_RESULT_DIR)

DEVICE: cuda
CURRENT_DIR: E:\Programming\Python\DeepLearning\比赛\ConvNeXt 方案
PROJECT_ROOT: E:\Programming\Python\DeepLearning\比赛
IMAGE_DIR: E:\Programming\Python\DeepLearning\比赛\images
SOURCE_SPLIT_DIR: E:\Programming\Python\DeepLearning\比赛\fire_splits
MODEL_SAVE_DIR: E:\Programming\Python\DeepLearning\比赛\ConvNeXt 方案\models
ATTENTION_RESULT_DIR: E:\Programming\Python\DeepLearning\比赛\ConvNeXt 方案\attention_results


In [2]:
# =============
# Transform、模型加载、Attention 工具函数
# =============

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def load_threshold(default_threshold=DEFAULT_THRESHOLD):
    if THRESHOLD_JSON.exists():
        with open(THRESHOLD_JSON, "r", encoding="utf-8") as f:
            info = json.load(f)

        threshold = float(info["best_threshold"])
        print("已读取自动搜索 threshold:", threshold)
        print("threshold 文件:", THRESHOLD_JSON)
        return threshold

    print("未找到自动 threshold 文件，使用默认 threshold:", default_threshold)
    return float(default_threshold)


def discover_model_paths(checkpoint_type="best"):
    pattern = f"{MODEL_PREFIX}_*_{checkpoint_type}.pth"

    model_paths = sorted(MODEL_SAVE_DIR.glob(pattern))

    if len(model_paths) == 0:
        raise FileNotFoundError(
            f"未找到模型权重。搜索路径: {MODEL_SAVE_DIR / pattern}"
        )

    return model_paths


def load_models(checkpoint_type="best"):
    model_paths = discover_model_paths(
        checkpoint_type=checkpoint_type
    )

    models_list = []

    for model_path in model_paths:
        model, checkpoint = load_fire_model_from_checkpoint(
            checkpoint_path=model_path,
            device=DEVICE,
            finetune_mode=FINETUNE_MODE,
            dropout=DROPOUT,
            max_pool_scale=MAX_POOL_SCALE,
            strict=True
        )

        models_list.append(model)

        print("Loaded:", model_path)

    return models_list, model_paths


def resolve_image_path(input_image_path):
    p = Path(str(input_image_path)).expanduser()

    if p.exists():
        return p

    p2 = PROJECT_ROOT / p
    if p2.exists():
        return p2

    p3 = IMAGE_DIR / p.name
    if p3.exists():
        return p3

    raise FileNotFoundError(f"输入图片不存在: {input_image_path}")


def get_image_path_from_row(row):
    if "path" in row and pd.notna(row["path"]):
        p = Path(str(row["path"]))
        if p.exists():
            return p

    p = IMAGE_DIR / str(row["filename"])

    if p.exists():
        return p

    raise FileNotFoundError(f"找不到图片: {row['filename']}")


@torch.no_grad()
def predict_one_image_all_models(
    models_list,
    image_tensor
):
    probs = []

    for model in models_list:
        logit = model(image_tensor).squeeze()
        prob = torch.sigmoid(logit).item()
        probs.append(prob)

    probs = np.array(probs, dtype=np.float32)
    mean_prob = float(probs.mean())

    return probs, mean_prob


@torch.no_grad()
def get_attention_map_one_model(
    model,
    image_tensor,
    raw_h,
    raw_w
):
    logit, attention_map = model(
        image_tensor,
        return_attention=True
    )

    attention_up = F.interpolate(
        attention_map,
        size=(raw_h, raw_w),
        mode="bilinear",
        align_corners=False
    )

    attention_np = attention_up.squeeze().detach().cpu().numpy()

    return attention_np


@torch.no_grad()
def get_attention_map(
    models_list,
    image_tensor,
    raw_h,
    raw_w,
    use_mean_attention=True
):
    if not use_mean_attention:
        return get_attention_map_one_model(
            model=models_list[0],
            image_tensor=image_tensor,
            raw_h=raw_h,
            raw_w=raw_w
        )

    attention_maps = []

    for model in models_list:
        attention_map = get_attention_map_one_model(
            model=model,
            image_tensor=image_tensor,
            raw_h=raw_h,
            raw_w=raw_w
        )

        attention_maps.append(attention_map)

    mean_attention_map = np.mean(
        np.stack(attention_maps, axis=0),
        axis=0
    )

    return mean_attention_map


def normalize_attention_map(attention_map):
    attention_min = float(attention_map.min())
    attention_max = float(attention_map.max())

    if attention_max - attention_min < 1e-8:
        return np.zeros_like(attention_map)

    return (attention_map - attention_min) / (attention_max - attention_min)


def save_attention_overlay(
    raw_image,
    attention_map,
    save_path,
    title
):
    attention_map = normalize_attention_map(attention_map)

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.imshow(raw_image)
    ax.imshow(
        attention_map,
        alpha=0.45,
        cmap="jet"
    )

    ax.set_title(title)
    ax.axis("off")

    fig.tight_layout()
    fig.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

In [4]:
# =============
# 单张图片 Attention map
# =============

ATTENTION_IMAGE_PATH = r"../images/20250526_firesmoke_03261.jpg"

CHECKPOINT_TYPE = "best"
USE_MEAN_ATTENTION = True

SINGLE_ATTENTION_DIR = ATTENTION_RESULT_DIR / "single_image"
SINGLE_ATTENTION_DIR.mkdir(parents=True, exist_ok=True)

models_list, model_paths = load_models(
    checkpoint_type=CHECKPOINT_TYPE
)

image_path = resolve_image_path(
    ATTENTION_IMAGE_PATH
)

raw_image = Image.open(image_path).convert("RGB")
raw_w, raw_h = raw_image.size

image_tensor = eval_transform(raw_image).unsqueeze(0).to(DEVICE)

probs_each, mean_prob = predict_one_image_all_models(
    models_list=models_list,
    image_tensor=image_tensor
)

attention_map = get_attention_map(
    models_list=models_list,
    image_tensor=image_tensor,
    raw_h=raw_h,
    raw_w=raw_w,
    use_mean_attention=USE_MEAN_ATTENTION
)

save_path = SINGLE_ATTENTION_DIR / f"attention_{image_path.stem}_prob_{mean_prob:.4f}.png"

title = f"prob_fire = {mean_prob:.4f}"

save_attention_overlay(
    raw_image=raw_image,
    attention_map=attention_map,
    save_path=save_path,
    title=title
)

print("\n输入图片:")
print(image_path)

print("\n每个模型 prob_fire:")
for i, prob in enumerate(probs_each):
    print(f"model_{i:02d}:", round(float(prob), 6))

print("\nensemble prob_fire:", round(mean_prob, 6))

print("\nAttention map 已保存:")
print(save_path)

Loaded: E:\Programming\Python\DeepLearning\比赛\ConvNeXt 方案\models\fire_convnext_tiny_attention_00_best.pth

输入图片:
..\images\20250526_firesmoke_03261.jpg

每个模型 prob_fire:
model_00: 0.740645

ensemble prob_fire: 0.740645

Attention map 已保存:
E:\Programming\Python\DeepLearning\比赛\ConvNeXt 方案\attention_results\single_image\attention_20250526_firesmoke_03261_prob_0.7406.png


In [3]:
# =============
# 生成所有错误预测样本的 Attention map
# 只考虑原始真实图片，不考虑增强图、manual_no_fire_crops、自动裁剪图
# =============

ATTENTION_CSV = VAL_CSV
# ATTENTION_CSV = TEST_CSV
# ATTENTION_CSV = SOURCE_ENSEMBLE_DIR / "ensemble_train_00.csv"

CHECKPOINT_TYPE = "best"

ATTENTION_THRESHOLD = load_threshold(
    default_threshold=DEFAULT_THRESHOLD
)

USE_ENSEMBLE_PROB = True
USE_MEAN_ATTENTION = True

MAX_SAVE_ERRORS = None

WRONG_ATTENTION_DIR = ATTENTION_RESULT_DIR / "wrong_prediction_attention_maps_original_only"
FALSE_POSITIVE_DIR = WRONG_ATTENTION_DIR / "false_positive_label0_pred1"
FALSE_NEGATIVE_DIR = WRONG_ATTENTION_DIR / "false_negative_label1_pred0"

FALSE_POSITIVE_DIR.mkdir(parents=True, exist_ok=True)
FALSE_NEGATIVE_DIR.mkdir(parents=True, exist_ok=True)

ERROR_CSV_PATH = WRONG_ATTENTION_DIR / "wrong_prediction_attention_summary_original_only.csv"

models_list, model_paths = load_models(
    checkpoint_type=CHECKPOINT_TYPE
)

df = pd.read_csv(ATTENTION_CSV)
df["label"] = df["label"].astype(int)

# 只保留原始真实图片
before_n = len(df)

if "is_augmented" in df.columns:
    df = df[df["is_augmented"] == False].copy()

if "is_manual_crop" in df.columns:
    df = df[df["is_manual_crop"] == False].copy()

if "source_type" in df.columns:
    df = df[df["source_type"] == "original_real_image"].copy()

if "aug_type" in df.columns:
    df = df[df["aug_type"] == "original"].copy()

df = df.reset_index(drop=True)

print("\nAttention 数据集:")
print(ATTENTION_CSV)
print("过滤前样本数:", before_n)
print("过滤后原始真实图片数:", len(df))
print("标签分布:")
print(df["label"].value_counts().sort_index())
print("ATTENTION_THRESHOLD:", ATTENTION_THRESHOLD)

wrong_rows = []
saved_count = 0

for idx, row in df.iterrows():
    label = int(row["label"])
    filename = str(row["filename"])

    image_path = get_image_path_from_row(row)

    raw_image = Image.open(image_path).convert("RGB")
    raw_w, raw_h = raw_image.size

    image_tensor = eval_transform(raw_image).unsqueeze(0).to(DEVICE)

    probs_each, mean_prob = predict_one_image_all_models(
        models_list=models_list,
        image_tensor=image_tensor
    )

    if USE_ENSEMBLE_PROB:
        prob_fire = mean_prob
    else:
        prob_fire = float(probs_each[0])

    pred = int(prob_fire >= ATTENTION_THRESHOLD)

    if pred == label:
        continue

    if label == 0 and pred == 1:
        error_type = "false_positive"
        save_dir = FALSE_POSITIVE_DIR
    elif label == 1 and pred == 0:
        error_type = "false_negative"
        save_dir = FALSE_NEGATIVE_DIR
    else:
        error_type = "unknown_error"
        save_dir = WRONG_ATTENTION_DIR

    attention_map = get_attention_map(
        models_list=models_list,
        image_tensor=image_tensor,
        raw_h=raw_h,
        raw_w=raw_w,
        use_mean_attention=USE_MEAN_ATTENTION
    )

    save_name = (
        f"{error_type}_"
        f"idx{idx:04d}_"
        f"label{label}_pred{pred}_"
        f"prob{prob_fire:.4f}_"
        f"{Path(filename).stem}.png"
    )

    save_path = save_dir / save_name

    title = (
        f"{error_type} | "
        f"label={label}, pred={pred}, "
        f"prob_fire={prob_fire:.4f}, "
        f"threshold={ATTENTION_THRESHOLD:.2f}"
    )

    save_attention_overlay(
        raw_image=raw_image,
        attention_map=attention_map,
        save_path=save_path,
        title=title
    )

    wrong_record = {
        "index": idx,
        "filename": filename,
        "path": str(image_path),
        "label": label,
        "pred": pred,
        "prob_fire": prob_fire,
        "threshold": ATTENTION_THRESHOLD,
        "error_type": error_type,
        "attention_path": str(save_path),
        "image_width": raw_w,
        "image_height": raw_h,
        "is_augmented": row["is_augmented"] if "is_augmented" in row.index else None,
        "is_manual_crop": row["is_manual_crop"] if "is_manual_crop" in row.index else None,
        "source_type": row["source_type"] if "source_type" in row.index else None,
        "aug_type": row["aug_type"] if "aug_type" in row.index else None,
    }

    for model_id, prob_i in enumerate(probs_each):
        wrong_record[f"model_{model_id:02d}_prob_fire"] = float(prob_i)

    wrong_rows.append(wrong_record)

    saved_count += 1

    if saved_count % 10 == 0:
        print(f"已保存错误 attention 数量: {saved_count}")

    if MAX_SAVE_ERRORS is not None and saved_count >= MAX_SAVE_ERRORS:
        print(f"达到 MAX_SAVE_ERRORS={MAX_SAVE_ERRORS}，提前停止。")
        break

wrong_df = pd.DataFrame(wrong_rows)

wrong_df.to_csv(
    ERROR_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n处理完成")
print("错误预测样本数量:", len(wrong_df))

if len(wrong_df) > 0:
    print("\n错误类型统计:")
    print(wrong_df["error_type"].value_counts())

    print("\nprob_fire 统计:")
    print(wrong_df["prob_fire"].describe())

print("\nAttention 结果保存目录:")
print(WRONG_ATTENTION_DIR)

print("\nFalse Positive 目录:")
print(FALSE_POSITIVE_DIR)

print("\nFalse Negative 目录:")
print(FALSE_NEGATIVE_DIR)

print("\n错误样本汇总 CSV:")
print(ERROR_CSV_PATH)

未找到自动 threshold 文件，使用默认 threshold: 0.6
Loaded: E:\Programming\Python\DeepLearning\比赛\ConvNeXt 方案\models\fire_convnext_tiny_attention_00_best.pth

Attention 数据集:
E:\Programming\Python\DeepLearning\比赛\fire_splits\val_fixed_real_only.csv
过滤前样本数: 100
过滤后原始真实图片数: 100
标签分布:
label
0    50
1    50
Name: count, dtype: int64
ATTENTION_THRESHOLD: 0.6
已保存错误 attention 数量: 10
已保存错误 attention 数量: 20
已保存错误 attention 数量: 30

处理完成
错误预测样本数量: 32

错误类型统计:
error_type
false_positive    21
false_negative    11
Name: count, dtype: int64

prob_fire 统计:
count    32.000000
mean      0.638285
std       0.240965
min       0.183864
25%       0.394998
50%       0.707145
75%       0.820312
max       0.937308
Name: prob_fire, dtype: float64

Attention 结果保存目录:
E:\Programming\Python\DeepLearning\比赛\ConvNeXt 方案\attention_results\wrong_prediction_attention_maps_original_only

False Positive 目录:
E:\Programming\Python\DeepLearning\比赛\ConvNeXt 方案\attention_results\wrong_prediction_attention_maps_original_only\false_positive_l

In [ ]:
# =============
# 在 Attention notebook 中生成 _hard_weighted.csv
# =============
# 作用：
# 用当前 best 模型预测训练集；
# 找出 label=0 但 prob_fire 高的 no_fire 样本；
# 给这些 hard no_fire 提高 sample_weight；
# 保存为：
# ../fire_splits/ensemble_splits/ensemble_train_00_hard_weighted.csv
#
# 注意：
# 1. 本 cell 建议放在 Attention notebook 的模型加载函数之后。
# 2. 需要前面已经定义：
#    - CURRENT_DIR
#    - PROJECT_ROOT
#    - IMAGE_DIR
#    - SOURCE_SPLIT_DIR
#    - eval_transform
#    - load_models()
#    - get_image_path_from_row()
#    - predict_one_image_all_models()
# 3. 生成后，训练 notebook 会优先读取 hard_weighted 版本。

import math
from pathlib import Path

SOURCE_ENSEMBLE_DIR = SOURCE_SPLIT_DIR / "ensemble_splits"

HARD_WEIGHT_RESULT_DIR = ATTENTION_RESULT_DIR / "hard_negative_mining"
HARD_WEIGHT_RESULT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_TYPE = "best"

# hard negative 阈值与权重
HARD_NEG_PROB_1 = 0.60
HARD_NEG_PROB_2 = 0.75
HARD_NEG_PROB_3 = 0.90

NORMAL_WEIGHT = 1.0
HARD_WEIGHT_1 = 2.0
HARD_WEIGHT_2 = 3.0
HARD_WEIGHT_3 = 4.0

# 是否覆盖已有 _hard_weighted.csv
OVERWRITE_HARD_WEIGHTED_CSV = True

# 是否只处理指定训练 CSV
# None 表示自动处理所有 ensemble_train_xx.csv
ONLY_ENSEMBLE_ID = None
# ONLY_ENSEMBLE_ID = 0


def list_original_ensemble_train_csvs():
    if ONLY_ENSEMBLE_ID is not None:
        csv_path = SOURCE_ENSEMBLE_DIR / f"ensemble_train_{ONLY_ENSEMBLE_ID:02d}.csv"

        if not csv_path.exists():
            raise FileNotFoundError(f"未找到训练 CSV: {csv_path}")

        return [csv_path]

    csv_paths = sorted(SOURCE_ENSEMBLE_DIR.glob("ensemble_train_*.csv"))

    # 排除已经生成的 hard_weighted 版本
    csv_paths = [
        p for p in csv_paths
        if not p.name.endswith("_hard_weighted.csv")
    ]

    if len(csv_paths) == 0:
        raise FileNotFoundError(
            f"未找到 ensemble_train_xx.csv，目录: {SOURCE_ENSEMBLE_DIR}"
        )

    return csv_paths


@torch.no_grad()
def predict_train_csv_probs(train_csv, models_list):
    train_df = pd.read_csv(train_csv)
    train_df["label"] = train_df["label"].astype(int)

    all_records = []

    for idx, row in train_df.iterrows():
        image_path = get_image_path_from_row(row)

        raw_image = Image.open(image_path).convert("RGB")
        image_tensor = eval_transform(raw_image).unsqueeze(0).to(DEVICE)

        probs_each, mean_prob = predict_one_image_all_models(
            models_list=models_list,
            image_tensor=image_tensor
        )

        record = {
            "index": idx,
            "filename": str(row["filename"]),
            "path": str(image_path),
            "label": int(row["label"]),
            "prob_fire": float(mean_prob)
        }

        for model_id, prob_i in enumerate(probs_each):
            record[f"model_{model_id:02d}_prob_fire"] = float(prob_i)

        all_records.append(record)

        if (idx + 1) % 50 == 0:
            print(f"{train_csv.name} | 已预测 {idx + 1}/{len(train_df)}")

    pred_df = pd.DataFrame(all_records)

    return train_df, pred_df


def add_hard_negative_weights(train_df, pred_df):
    out_df = train_df.copy()

    if len(out_df) != len(pred_df):
        raise RuntimeError(
            f"训练 CSV 行数与预测行数不一致: train={len(out_df)}, pred={len(pred_df)}"
        )

    out_df["prob_fire"] = pred_df["prob_fire"].values
    out_df["sample_weight"] = NORMAL_WEIGHT

    hard_mask_1 = (
        (out_df["label"].astype(int) == 0) &
        (out_df["prob_fire"] >= HARD_NEG_PROB_1)
    )

    hard_mask_2 = (
        (out_df["label"].astype(int) == 0) &
        (out_df["prob_fire"] >= HARD_NEG_PROB_2)
    )

    hard_mask_3 = (
        (out_df["label"].astype(int) == 0) &
        (out_df["prob_fire"] >= HARD_NEG_PROB_3)
    )

    out_df.loc[hard_mask_1, "sample_weight"] = HARD_WEIGHT_1
    out_df.loc[hard_mask_2, "sample_weight"] = HARD_WEIGHT_2
    out_df.loc[hard_mask_3, "sample_weight"] = HARD_WEIGHT_3

    summary = {
        "total": len(out_df),
        "no_fire_total": int((out_df["label"].astype(int) == 0).sum()),
        "fire_total": int((out_df["label"].astype(int) == 1).sum()),
        f"hard_no_fire_prob_ge_{HARD_NEG_PROB_1}": int(hard_mask_1.sum()),
        f"hard_no_fire_prob_ge_{HARD_NEG_PROB_2}": int(hard_mask_2.sum()),
        f"hard_no_fire_prob_ge_{HARD_NEG_PROB_3}": int(hard_mask_3.sum()),
        "normal_weight": NORMAL_WEIGHT,
        "hard_weight_1": HARD_WEIGHT_1,
        "hard_weight_2": HARD_WEIGHT_2,
        "hard_weight_3": HARD_WEIGHT_3
    }

    return out_df, summary


models_list, model_paths = load_models(
    checkpoint_type=CHECKPOINT_TYPE
)

train_csvs = list_original_ensemble_train_csvs()

print("将处理以下训练 CSV:")
for p in train_csvs:
    print(p)

all_summary_rows = []

for train_csv in train_csvs:
    ensemble_id_text = train_csv.stem.replace("ensemble_train_", "")
    save_csv = SOURCE_ENSEMBLE_DIR / f"{train_csv.stem}_hard_weighted.csv"

    if save_csv.exists() and not OVERWRITE_HARD_WEIGHTED_CSV:
        print(f"\n已存在，跳过: {save_csv}")
        continue

    print("\n" + "=" * 80)
    print("开始处理:", train_csv)

    train_df, pred_df = predict_train_csv_probs(
        train_csv=train_csv,
        models_list=models_list
    )

    hard_df, summary = add_hard_negative_weights(
        train_df=train_df,
        pred_df=pred_df
    )

    hard_df.to_csv(
        save_csv,
        index=False,
        encoding="utf-8-sig"
    )

    pred_save_csv = HARD_WEIGHT_RESULT_DIR / f"{train_csv.stem}_pred_probs.csv"
    pred_df.to_csv(
        pred_save_csv,
        index=False,
        encoding="utf-8-sig"
    )

    summary["train_csv"] = str(train_csv)
    summary["hard_weighted_csv"] = str(save_csv)
    summary["pred_prob_csv"] = str(pred_save_csv)

    all_summary_rows.append(summary)

    print("\n生成完成:")
    print(save_csv)

    print("\n样本权重分布:")
    print(hard_df["sample_weight"].value_counts().sort_index())

    print("\nHard no_fire 数量:")
    print(f"prob_fire >= {HARD_NEG_PROB_1}: {summary[f'hard_no_fire_prob_ge_{HARD_NEG_PROB_1}']}")
    print(f"prob_fire >= {HARD_NEG_PROB_2}: {summary[f'hard_no_fire_prob_ge_{HARD_NEG_PROB_2}']}")
    print(f"prob_fire >= {HARD_NEG_PROB_3}: {summary[f'hard_no_fire_prob_ge_{HARD_NEG_PROB_3}']}")

    print("\nTop 30 hard no_fire:")
    display_cols = [
        "filename",
        "label",
        "prob_fire",
        "sample_weight"
    ]

    extra_cols = [
        c for c in [
            "is_augmented",
            "is_manual_crop",
            "aug_type",
            "source_filename"
        ]
        if c in hard_df.columns
    ]

    display_cols = display_cols + extra_cols

    print(
        hard_df[hard_df["label"].astype(int) == 0]
        .sort_values("prob_fire", ascending=False)
        [display_cols]
        .head(30)
    )

summary_df = pd.DataFrame(all_summary_rows)

summary_save_csv = HARD_WEIGHT_RESULT_DIR / "hard_negative_mining_summary.csv"
summary_df.to_csv(
    summary_save_csv,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 80)
print("全部处理完成")
print("summary 保存到:")
print(summary_save_csv)

print("\n汇总:")
print(summary_df)